In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, hour, to_timestamp, avg, desc

spark = SparkSession.builder \
    .appName("AirQualityAnalysis") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .getOrCreate()

print("Spark Session Berhasil Dibuat")

In [ ]:
df = spark.read.json("hdfs://localhost:9000/data/airquality/api/*.json")

df.printSchema()
df.show(5)

In [ ]:
df_category = df.withColumn("status", 
    when(col("value") <= 50, "Baik")
    .when(col("value") <= 100, "Sedang")
    .otherwise("Tidak Sehat")
)

df_category.groupBy("status").count().show()

In [ ]:
df_time = df.withColumn("hour", hour(to_timestamp(col("timestamp"))))

avg_hourly = df_time.groupBy("hour").agg(avg("value").alias("avg_aqi")).orderBy("hour")
avg_hourly.show()

In [ ]:
ranking_city = df.groupBy("city") \
    .agg(avg("value").alias("average_pollution")) \
    .orderBy(desc("average_pollution"))

ranking_city.show()

In [ ]:
import os

# Pastikan folder tujuan ada
output_path = "../dashboard/data"
if not os.path.exists(output_path):
    os.makedirs(output_path)

ranking_city.toPandas().to_json(f"{output_path}/spark_results.json", orient="records")

print("Hasil analisis telah diekspor ke dashboard/data/spark_results.json")